# Acoustic machine-failure detection: quickstart

Simulate a bearing with an outer-race defect, find the resonance band with the kurtogram, and read the defect frequency from the envelope spectrum. No data download; under a minute.

Everything here calls the public physics core in `src/`. Run the cells top to bottom.

In [ ]:
# Setup: works in Google Colab and in a local clone of the repository.
import os, sys
if not os.path.isdir("src"):
    !git clone -q https://github.com/drdmitrymikhaylov/acousticpinn-machine-failure-prediction.git
    %cd acousticpinn-machine-failure-prediction
!pip -q install numpy scipy matplotlib
sys.path.insert(0, "src")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kinematics import BEARING_6205
from bearing_sim import simulate
from envelope import kurtogram, envelope_spectrum, harmonic_statistic

fr = 29.95                       # shaft rate, Hz (about 1800 rpm)
freqs = BEARING_6205.frequencies(fr)
print({k: round(float(v), 1) for k, v in freqs.items()})

## 1. A defect buried in noise

Outer-race defect, impulse train 9 dB below the noise, running-speed harmonics 12 dB above it. In the raw spectrum the defect is invisible.

In [ ]:
f_def = freqs["BPFO"]
fs = 48000.0
x, meta = simulate(f_def, fr, fs=fs, duration=1.0, snr_db=-9.0, seed=1)
F = np.fft.rfftfreq(len(x), 1 / fs)
X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
plt.figure(figsize=(6, 3)); plt.semilogy(F, X); plt.xlim(0, 2000)
plt.xlabel("frequency (Hz)"); plt.title("Raw spectrum"); plt.tight_layout(); plt.show()

## 2. Kurtogram, then the envelope spectrum

The kurtogram picks the band where the impulses live; demodulating that band puts the defect frequency and its harmonics on display.

In [ ]:
table, best = kurtogram(x, fs, n_levels=6, bw_min=8 * f_def)
level, fc, bw, k = best
band = (fc - bw / 2, fc + bw / 2)
print(f"kurtogram band: {band[0]:.0f} to {band[1]:.0f} Hz (kurtosis {k:.2f})")

fe, E = envelope_spectrum(x, fs, band=band)
stat, peaks, floors = harmonic_statistic(fe, E, f_def, n_harm=6)
print(f"harmonic statistic: {stat:.1f} dB above the local floor")

plt.figure(figsize=(6, 3.5)); plt.plot(fe, E)
for h in range(1, 7):
    plt.axvline(h * f_def, color="r", ls=":", lw=1)
plt.xlim(0, 7 * f_def)
plt.xlabel("envelope frequency (Hz)")
plt.title(f"Envelope spectrum, red lines = harmonics of BPFO ({f_def:.1f} Hz)")
plt.tight_layout(); plt.show()

Change `snr_db` to see where detection fails, or set `load_modulation=0.8` with `freqs["BPFI"]` for an inner-race defect. The README reports these limits over 90 records per condition.